In [1]:
from requests import get, post
from sys import exit
from pprint import pprint
import pandas as pd

In [2]:
HOST = "https://api.chartmetric.com"
TOKEN = "eVCH88EH3jriTh1ps2OpDtXeDMwlPPM7GY9y2J8jFkBY3RuLtx19DIXzC1sn0Ygm"
res = post(f"{HOST}/api/token", json={"refreshtoken": TOKEN})

if res.status_code != 200:
    print(f"ERROR: received a {res.status_code} instead of 200 from /api/token")
    exit(1)

access_token = res.json()["token"]

def Get(endpoint, params=None):
    res = get(
        f"{HOST}{endpoint}",
        headers={"Authorization": f"Bearer {access_token}"},
        params=params
    )
    if res.status_code != 200:
        print("ERROR:", res.status_code, res.text)
        return None
    return res.json()

In [3]:
genres = Get("/api/genre")
genres

{'obj': [{'id': 501120, 'name': 'pop'},
  {'id': 501121, 'name': 'hip-hop/rap'},
  {'id': 501122, 'name': 'rock'},
  {'id': 501123, 'name': 'metal'},
  {'id': 501124, 'name': 'electronic'},
  {'id': 501125, 'name': 'r&b/soul'},
  {'id': 501126, 'name': 'classical'},
  {'id': 501127, 'name': 'jazz'},
  {'id': 501128, 'name': 'blues'},
  {'id': 501129, 'name': 'devotional & spiritual'},
  {'id': 501130, 'name': 'folk'},
  {'id': 501131, 'name': 'country'},
  {'id': 501132, 'name': 'alternative'},
  {'id': 501133, 'name': 'eras'},
  {'id': 501134, 'name': 'ambient'},
  {'id': 501135, 'name': 'psychedelic'},
  {'id': 501136, 'name': 'spoken / comedy'},
  {'id': 501137, 'name': 'vocal'},
  {'id': 501138, 'name': 'singer/songwriter'},
  {'id': 501139, 'name': 'soundtrack'},
  {'id': 501140, 'name': 'instrumental'},
  {'id': 501141, 'name': 'holiday'},
  {'id': 501142, 'name': 'easy listening'},
  {'id': 501143, 'name': 'worldbeat'},
  {'id': 501144, 'name': 'industrial'},
  {'id': 501145, 'n

In [4]:
# STEP 1: USER INPUT

country = "CL"
genre_id = None
mood = "celebratory"

metric = "sp_monthly_listeners"
min_pop = 100000
max_pop = 200000

gender = "female"
min_gender_pct = 0.50

age = ["18", "24"]
min_age_pct = 0.20

In [5]:
# STEP 2: INITIAL ARTIST FILTER

params = {
    "min": min_pop,
    "max": max_pop,
    "code2": country,
    "limit": 50
}

artist_list = Get(f"/api/artist/{metric}/list", params=params)
pprint(artist_list)

artists = artist_list["obj"]["data"]

{'obj': {'data': [{'bandsintown_followers': None,
                   'chartmetric_artist_id': 8908848,
                   'code2': 'CL',
                   'cpp_rank': 125044,
                   'deezer_fans': None,
                   'facebook_likes': None,
                   'facebook_talks': None,
                   'genres': 'chilean, electronic, latin, latin hip-hop/rap, '
                             'latin pop, perreo, reggaeton, reggaeton chileno, '
                             'urbano latino',
                   'instagram_followers': 9934,
                   'name': 'GUNTTER',
                   'rank_eg': 119596,
                   'rank_fb': 125639,
                   'signed': False,
                   'soundcloud_followers': 24,
                   'sp_where_people_listen': [{'code2': 'mx',
                                               'listeners': 26673,
                                               'name': 'Mexico City'},
                                              {

In [6]:
# STEP 3: FILTER BY AUDIENCE DEMOGRAPHICS

# Mapping age range
AGE_KEY_MAP = {
    ("13", "17"): "13_17",
    ("18", "24"): "18_24",
    ("25", "34"): "25_34",
    ("35", "44"): "35_44",
    ("45", "64"): "45_64",
    ("65", "+"):  "65_",
}

def get_demographics(artist_id):
    return Get(f"/api/artist/{artist_id}/social-audience-stats", params={
        "domain": "instagram",
        "audienceType": "followers",
        "statsType": "demographic"
    })

def matches_demographics(artist):
    artist_id = artist.get("chartmetric_artist_id")
    artist_name = artist.get("name", "Unknown")

    data = get_demographics(artist_id)

    if not data or not data.get("obj"):
        print(f"  ERROR: No demographic data for {artist_name}")
        return False, {}

    obj = data["obj"][0]

    # Gender check
    gender_pct = obj.get(gender, 0)

    # Age check
    age_suffix = AGE_KEY_MAP.get(tuple(age))
    if not age_suffix:
        print(f"  ERROR:  Age range {age} not recognized.")
        return False, {}

    # Total share of that age group (male + female combined)
    age_key = f"ages_{age_suffix}"
    age_pct = obj.get(age_key, 0)

    demographics = {
        "gender_pct": gender_pct,
        "age_pct":    age_pct,
        "raw":        obj
    }

    match = (gender_pct >= min_gender_pct) and (age_pct >= min_age_pct)
    return match, demographics

In [7]:
# ── Run the filter ──

matched_artists = []

for artist in artists:
    name = artist.get("name", "Unknown")
    print(f"Checking: {name}...")

    is_match, demo = matches_demographics(artist)

    if is_match:
        matched_artists.append({**artist, "demographics": demo})
        print(f"  ✅ MATCH — {gender}: {demo['gender_pct']:.1%}, age {age[0]}-{age[1]}: {demo['age_pct']:.1%}")
    else:
        if demo:
            print(f"  ❌ No match — {gender}: {demo['gender_pct']:.1%}, age {age[0]}-{age[1]}: {demo['age_pct']:.1%}")

print(f"\n── {len(matched_artists)} artists matched your demographic criteria ──\n")
for a in matched_artists:
    d = a["demographics"]
    print(f"  🎵 {a['name']} — {gender}: {d['gender_pct']:.1%}, age {age[0]}-{age[1]}: {d['age_pct']:.1%}")

Checking: GUNTTER...
  ❌ No match — female: 46.7%, age 18-24: 44.5%
Checking: Poison Kid...
  ERROR: No demographic data for Poison Kid
Checking: Fármacos...
  ✅ MATCH — female: 60.1%, age 18-24: 37.5%
Checking: Ober...
  ERROR: No demographic data for Ober
Checking: Vishoko...
  ERROR: No demographic data for Vishoko
Checking: El Shaaki...
  ❌ No match — female: 35.7%, age 18-24: 35.1%
Checking: La Sonora De Tommy Rey...
  ❌ No match — female: 37.7%, age 18-24: 15.3%
Checking: ataquemos...
  ❌ No match — female: 39.7%, age 18-24: 49.1%
Checking: Chocolate Blanco...
  ERROR: No demographic data for Chocolate Blanco
Checking: Azerbeats...
  ❌ No match — female: 19.0%, age 18-24: 42.0%
Checking: Diego Lorenzini...
  ✅ MATCH — female: 55.0%, age 18-24: 34.7%
Checking: Los Galos...
  ERROR: No demographic data for Los Galos
Checking: Axl Boore...
  ERROR: No demographic data for Axl Boore
Checking: Sniper SP...
  ERROR: No demographic data for Sniper SP
Checking: La Descendencia Chilena...